<a href="https://colab.research.google.com/github/Lee-Minsoo-97/Sales-Data-Prediction/blob/gemini_original/Forecasting_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Forecasting**



## **Sales Data Loading**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

# 1. 데이터 로딩 (Data Loading)
# ------------------------------------
path = r'/content/drive/MyDrive/Colab Notebooks/iHerb_Sales_Data' # 본인의 파일 경로 확인
all_files = glob.glob(path + "/*.csv")
li = []

print(f"총 {len(all_files)}개의 파일을 처리합니다.")

for filename in all_files:
    try:
        df = pd.read_csv(filename, index_col=None, header=0)

        # 동적으로 컬럼 이름 찾기 (가장 마지막 컬럼을 'Sales'로 인식)
        status_col = [col for col in df.columns if col.endswith('_Status')][0]
        sales_col = df.columns[-1]

        # 컬럼 이름을 'Sales', 'Status'로 통일
        df.rename(columns={sales_col: 'Sales', status_col: 'Status'}, inplace=True)

        # 파일 이름에서 날짜 정보 추출하여 'Date' 컬럼 생성
        date_part = filename.split('/')[-1].split('_')[0]
        df['Date'] = pd.to_datetime(date_part, format='%Y.%m')

        li.append(df)
    except Exception as e:
        print(f"파일 처리 중 오류 발생 {filename}: {e}")

# 모든 파일을 하나로 합치기
master_df = pd.concat(li, axis=0, ignore_index=True)


# 2. 데이터 정제 및 변환 (Data Cleaning & Transformation)
# ------------------------------------
# 'Sales' 컬럼을 숫자로 변환 (숫자가 아닌 값은 0으로 처리)
master_df['Sales'] = pd.to_numeric(master_df['Sales'], errors='coerce').fillna(0)

# 'Date'와 'SKU' 기준으로 데이터 정렬
master_df.sort_values(by=['Date', 'SKU'], inplace=True)

# 'UPC Code' 타입을 .0이 없는 정수형(Nullable Integer)으로 변환
master_df['UPC Code'] = master_df['UPC Code'].astype('Int64')


# ★★★ 새로 추가된 부분: UPC Code가 0인 불완전 데이터 제거 ★★★
print(f"\nUPC Code 클리닝 전 데이터 개수: {len(master_df)}")
rows_to_remove = len(master_df[master_df['UPC Code'] == 0])
print(f"제거될 데이터(UPC Code가 0) 개수: {rows_to_remove}")

# UPC Code가 0이 아닌 행들만 선택하여 master_df를 덮어쓰기
master_df = master_df[master_df['UPC Code'] != 0].copy()
# ---------------------------------------------------------


# 뒤죽박죽인 Index를 0부터 시작하도록 초기화 (필터링 후에 실행)
master_df.reset_index(drop=True, inplace=True)


# 3. 최종 결과 확인 (Final Verification)
# ------------------------------------
print("\n\n✅ --- 데이터 처리 완료 --- ✅")
print("\n--- 최종 데이터 샘플 (Head) ---")
print(master_df.head())
print("\n--- 최종 데이터 정보 (Info) ---")
master_df.info()

총 16개의 파일을 처리합니다.

UPC Code 클리닝 전 데이터 개수: 4673
제거될 데이터(UPC Code가 0) 개수: 28


✅ --- 데이터 처리 완료 --- ✅

--- 최종 데이터 샘플 (Head) ---
        SKU       UPC Code                           Product Description  \
0  APL21688  8804014216882   Hyaluron Shot, Ampoule, 3.38 fl oz (100 ml)   
1  APL21691  8804014216912     Hyaluron Shot, Toner, 7.43 fl oz (220 ml)   
2  APL21693  8804014216936  Hyaluron Shot, Emulsion, 4.39 fl oz (130 ml)   
3  APL21694  8804014216943      Hyaluron Shot, Cream, 2.02 fl oz (60 ml)   
4  APL21699  8804014216998    Peptide Shot, Ampoule, 3.38 fl oz (100 ml)   

  Brand Code Brand Name Status  Sales       Date  
0        APL    AMPLE:N   NoPO     11 2024-05-01  
1        APL    AMPLE:N   NoPO      5 2024-05-01  
2        APL    AMPLE:N   NoPO      8 2024-05-01  
3        APL    AMPLE:N   NoPO     23 2024-05-01  
4        APL    AMPLE:N   NoPO     55 2024-05-01  

--- 최종 데이터 정보 (Info) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4645 entries, 0 to 4644
Data columns

In [ ]:
master_df

,SKU,UPC Code,Product Description,Brand Code,Brand Name,Status,Sales,Date
0,APL21688,8804014216882,"Hyaluron Shot, Ampoule, 3.38 fl oz (100 ml)",APL,AMPLE:N,NoPO,11,2024-05-01
1,APL21691,8804014216912,"Hyaluron Shot, Toner, 7.43 fl oz (220 ml)",APL,AMPLE:N,NoPO,5,2024-05-01
2,APL21693,8804014216936,"Hyaluron Shot, Emulsion, 4.39 fl oz (130 ml)",APL,AMPLE:N,NoPO,8,2024-05-01
3,APL21694,8804014216943,"Hyaluron Shot, Cream, 2.02 fl oz (60 ml)",APL,AMPLE:N,NoPO,23,2024-05-01
4,APL21699,8804014216998,"Peptide Shot, Ampoule, 3.38 fl oz (100 ml)",APL,AMPLE:N,NoPO,55,2024-05-01
...,...,...,...,...,...,...,...,...
4640,TWA31351,8800276313512,"Glaze Bouncing Tint, 07 Chewy, 4.5 g",TWA,2aN,New Seller,5,2025-08-01
4641,TWA31352,8800276313529,"Glaze Bouncing Tint, 08 Rose Moon, 4.5 g",TWA,2aN,New Seller,6,2025-08-01
4642,TWA31353,8800276313536,"Glaze Bouncing Tint, 09 Candy Chew, 4.5 g",TWA,2aN,New Seller,8,2025-08-01
4643,TWA31354,8800276313543,"Glaze Bouncing Tint, 10 Cherry Bite, 4.5 g",TWA,2aN,New Seller,9,2025-08-01


In [ ]:
# Interactive Google Sheet

'''
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=master_df)
'''

'\nfrom google.colab import sheets\nsheet = sheets.InteractiveSheet(df=master_df)\n'

## **Sales EDA**

### 1. 월별 전체 판매량 추이

In [ ]:
import plotly.express as px

# 1. 월별 전체 판매량 데이터 준비 (기존과 동일)
monthly_sales = master_df.groupby('Date')['Sales'].sum().reset_index()

# 2. Plotly를 사용하여 인터랙티브 라인 차트 생성
fig = px.line(monthly_sales,
              x='Date',
              y='Sales',
              title='월별 전체 판매량 (Total Sales Quantity over Time)',
              markers=True, # 데이터 포인트에 마커 표시
              labels={'Date': '날짜', 'Sales': '총 판매량'})

# 차트 레이아웃을 깔끔하게 업데이트
fig.update_layout(title_x=0.5, xaxis_title='', yaxis_title='Sales Quantity')
fig.show()

### 2. SKU별 월간 판매량 추이

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
# 필요한 라이브러리 import
import plotly.graph_objects as go
from ipywidgets import widgets, HBox, VBox
from IPython.display import display

# --- 1. 위젯 및 변수 설정 ---

# 검색 기준 선택 라디오 버튼
search_mode_selector = widgets.RadioButtons(
    options=['SKU', 'UPC Code'],
    value='SKU',
    description='검색 기준:',
    disabled=False
)

# SKU와 UPC Code 목록 미리 준비 (문자열로 변환하여 일관성 유지)
all_skus = sorted(master_df['SKU'].unique())
all_upcs = sorted(master_df['UPC Code'].dropna().astype(str).unique())

# SKU 검색창 (자동완성 기능 포함)
search_box = widgets.Combobox(
    placeholder='SKU를 검색하거나 입력하세요...',
    options=all_skus, # 초기 옵션은 SKU 리스트
    description='검색:',
    ensure_option=True
)

# 버튼 생성
add_button = widgets.Button(description="그래프에 추가")
clear_button = widgets.Button(description="초기화")

# 그래프 위젯
fig = go.FigureWidget()
fig.update_layout(title='검색된 SKU 판매 추이', title_x=0.5, yaxis_title='Sales Quantity', legend_title_text='SKU', height=750, width=1000)

# 선택된 SKU를 저장할 리스트 (항상 SKU로 저장)
selected_skus_list = []


# --- 2. 기능 정의 ---

# 검색 기준 변경 시, 검색창의 자동완성 목록을 업데이트하는 함수
def on_search_mode_change(change):
    if change['new'] == 'SKU':
        search_box.options = all_skus
        search_box.placeholder = 'SKU를 검색하거나 입력하세요...'
    else: # 'UPC Code'
        search_box.options = all_upcs
        search_box.placeholder = 'UPC Code를 검색하거나 입력하세요...'

# '그래프에 추가' 버튼 기능 (SKU와 UPC 모두 처리)
def on_add_button_clicked(b):
    search_value = search_box.value
    sku_to_add = None

    # 검색 기준에 따라 SKU 찾기
    if search_mode_selector.value == 'SKU':
        if search_value in all_skus:
            sku_to_add = search_value
    else: # UPC Code로 검색
        try:
            # UPC Code에 해당하는 SKU 찾기
            found_sku = master_df[master_df['UPC Code'].astype(str) == search_value]['SKU'].iloc[0]
            sku_to_add = found_sku
        except (ValueError, IndexError):
            print(f"UPC Code '{search_value}'에 해당하는 SKU를 찾을 수 없습니다.")
            return

    # 유효성 검사 (중복, 최대 개수 등)
    if not sku_to_add: return
    if sku_to_add in selected_skus_list:
        print(f"SKU {sku_to_add}는 이미 추가되었습니다.")
        return
    if len(selected_skus_list) >= 10:
        print("최대 10개의 SKU만 추가할 수 있습니다.")
        return

    selected_skus_list.append(sku_to_add)
    update_plot()

# '초기화' 버튼 기능
def on_clear_button_clicked(b):
    selected_skus_list.clear()
    update_plot()

# 그래프 업데이트 함수
def update_plot():
    fig.data = []
    if not selected_skus_list:
        fig.update_layout(title='SKU 또는 UPC Code를 검색하여 그래프에 추가하세요')
    else:
        filtered_df = master_df[master_df['SKU'].isin(selected_skus_list)]
        for sku in selected_skus_list:
            sku_df = filtered_df[filtered_df['SKU'] == sku]
            fig.add_trace(go.Scatter(x=sku_df['Date'], y=sku_df['Sales'], mode='lines+markers', name=sku))
        fig.update_layout(title=f'선택된 SKU 판매 추이 ({len(selected_skus_list)}개)')
    search_box.value = ''

# --- 3. 위젯과 기능 연결 및 화면 표시 ---

# 라디오 버튼의 값이 바뀔 때마다 on_search_mode_change 함수 실행
search_mode_selector.observe(on_search_mode_change, names='value')

add_button.on_click(on_add_button_clicked)
clear_button.on_click(on_clear_button_clicked)

# 최종 UI 레이아웃
controls = HBox([search_mode_selector, search_box, add_button, clear_button])
dashboard = VBox([controls, fig])

display(dashboard)
update_plot()

## **PO Data Loading & Combining with Sales Data**

In [ ]:
# --- 4. PO 데이터 불러오기 및 통합 ---

# 파일 이름 정의
po_file_name = '/content/drive/MyDrive/Colab Notebooks/sps_data.csv'

try:
    # 1. PO 데이터 불러오기 및 기본 처리
    po_df = pd.read_csv(po_file_name)
    required_columns = ['PO Date', 'SKU', 'Qty Ordered']
    po_df_simple = po_df[required_columns].copy()
    po_df_simple['PO_Date_dt'] = pd.to_datetime(po_df_simple['PO Date'])
    print(f"✅ '{po_file_name}' 파일을 성공적으로 불러왔습니다.")

    # 2. 월별 데이터로 집계
    po_df_simple['YearMonth'] = po_df_simple['PO_Date_dt'].dt.to_period('M')
    monthly_po = po_df_simple.groupby(['SKU', 'YearMonth'])['Qty Ordered'].sum().reset_index()
    monthly_po.rename(columns={'Qty Ordered': 'PO_Quantity'}, inplace=True)
    print("✅ 월별 PO 데이터 집계를 완료했습니다.")

    # 3. master_df와 통합 (Merge)
    master_df['YearMonth'] = master_df['Date'].dt.to_period('M')
    df_merged = pd.merge(master_df, monthly_po, on=['SKU', 'YearMonth'], how='left')
    df_merged['PO_Quantity'].fillna(0, inplace=True)
    print("✅ 판매 데이터와 PO 데이터 통합을 완료했습니다.")

    # 최종 분석용 데이터프레임으로 지정
    df_features = df_merged.copy()

    print("\n--- 최종 통합 데이터 샘플 (PO_Quantity 컬럼 확인) ---")
    print(df_features[['Date', 'SKU', 'Sales', 'PO_Quantity']].head())
    print(f"\n통합 후 데이터 개수: {len(df_features)}")


except FileNotFoundError:
    print(f"❌ 오류: '{po_file_name}' 파일을 찾을 수 없습니다.")
except KeyError as e:
    print(f"❌ 오류: 파일에서 필요한 컬럼을 찾을 수 없습니다 -> {e}")

✅ '/content/drive/MyDrive/Colab Notebooks/sps_data.csv' 파일을 성공적으로 불러왔습니다.
✅ 월별 PO 데이터 집계를 완료했습니다.
✅ 판매 데이터와 PO 데이터 통합을 완료했습니다.

--- 최종 통합 데이터 샘플 (PO_Quantity 컬럼 확인) ---
        Date       SKU  Sales  PO_Quantity
0 2024-05-01  APL21688     11          0.0
1 2024-05-01  APL21691      5        120.0
2 2024-05-01  APL21693      8         96.0
3 2024-05-01  APL21694     23          0.0
4 2024-05-01  APL21699     55         60.0

통합 후 데이터 개수: 4645


/tmp/ipython-input-2533853125.py:23: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





## **Creating Lag Features**



시차 피처(Lag Feature)란 무엇인가?
시차 피처는 특정 시점의 데이터를 예측하기 위해, 과거의 데이터를 현재 행으로 가져와 나란히 놓는 기법입니다. 즉, 모델에게 '과거에 이랬으니, 현재는 이럴 것이다'라고 추론할 수 있는 핵심적인 **과거의 단서**를 제공하는 과정입니다.

### 왜 '판매량' 시차 피처가 필요한가? (Sales Lag)
1. 판매의 '관성' (Momentum) 포착
판매량에는 일종의 관성이 있습니다. 잘 팔리던 제품은 다음 달에도 잘 팔릴 가능성이 높고, 안 팔리던 제품은 계속 안 팔릴 가능성이 높습니다. 이를 통계적으로 자기상관(Autocorrelation)이라고 합니다. **판매량 시차 피처**는 모델에게 이러한 판매의 자연스러운 흐름과 관성을 직접적으로 학습시킵니다.

2. '고객 행동'의 현실 반영
판매 데이터는 **고객의 실제 행동**을 담고 있는 가장 정확한 현실 데이터입니다. PO 데이터가 놓칠 수 있는 유기적인 트렌드(입소문 등), 갑작스러운 판매량 급감(경쟁사 출시 등)과 같은 고객단의 현실을 모델에게 알려줍니다.



### 왜 'PO 수량' 시차 피처가 필요한가? (PO Lag)
1. 강력한 '선행 지표' (Leading Indicator)
PO는 실제 판매보다 1~2개월 먼저 발생하는 **선행 지표**입니다. 즉, 미래에 발생할 판매에 대한 **미리보기**와 같습니다. iHerb가 특정 제품을 대량 주문했다는 것은, 그들의 재고 계획과 미래 판매 예측에 대한 가장 강력한 신호입니다.

2. '공급망 계획'의 의도 파악
PO 데이터에는 iHerb의 프로모션 계획, 재고 정책 등 우리가 알 수 없는 내부적인 **계획과 의도**가 담겨 있습니다. 모델은 이 피처를 통해 iHerb의 계획을 학습하여 예측에 반영합니다.

In [ ]:
# 데이터를 SKU와 Date 순으로 정렬 (시차 계산의 정확성을 위해 필수)
df_features = df_features.sort_values(by=['SKU', 'Date'])

# SKU별로 그룹을 만들어, 각 SKU 내에서만 시차를 계산하도록 함
grouped = df_features.groupby('SKU')

# 1, 2, 3개월 전의 시차 피처를 반복문으로 생성
for lag in [1, 2, 3]:
    # 과거 판매량 (Sales)
    df_features[f'sales_lag_{lag}'] = grouped['Sales'].shift(lag)
    # 과거 PO 수량 (PO_Quantity)
    df_features[f'po_quantity_lag_{lag}'] = grouped['PO_Quantity'].shift(lag)

# 시차 계산으로 인해 생긴 초반 데이터의 빈 값(NaN)을 0으로 채움
df_features.fillna(0, inplace=True)

print("✅ 시차 피처 생성이 완료되었습니다.")
print("\n--- 새로 추가된 시차 피처 확인 ---")

# 특정 SKU('APL21691')를 예시로 결과 확인
# Date, 원본 데이터, 그리고 새로 생성된 시차 피처들을 함께 출력
print(df_features[df_features['SKU'] == 'APL21691'][
    ['Date', 'Sales', 'PO_Quantity', 'sales_lag_1', 'po_quantity_lag_1', 'sales_lag_2', 'po_quantity_lag_2']
].head())

✅ 시차 피처 생성이 완료되었습니다.

--- 새로 추가된 시차 피처 확인 ---
          Date  Sales  PO_Quantity  sales_lag_1  po_quantity_lag_1  \
1   2024-05-01      5        120.0          0.0                0.0   
210 2024-06-01      7          0.0          5.0              120.0   
433 2024-07-01      6          0.0          7.0                0.0   
672 2024-08-01      5          0.0          6.0                0.0   

     sales_lag_2  po_quantity_lag_2  
1            0.0                0.0  
210          0.0                0.0  
433          5.0              120.0  
672          7.0                0.0  


### **How 'df_features' looks like + 이동평균 feature 생성**

In [ ]:
print('df_features의 데이터 컬럼: \n\n',df_features.columns)

df_features의 데이터 컬럼: 

 Index(['SKU', 'UPC Code', 'Product Description', 'Brand Code', 'Brand Name',
       'Status', 'Sales', 'Date', 'YearMonth', 'PO_Quantity', 'sales_lag_1',
       'po_quantity_lag_1', 'sales_lag_2', 'po_quantity_lag_2', 'sales_lag_3',
       'po_quantity_lag_3'],
      dtype='object')


In [ ]:
print("--- df_features의 처음 10행 ---")
display(df_features.head(10))

print("\n--- df_features의 마지막 10행 ---")
display(df_features.tail(10))

--- df_features의 처음 10행 ---


,SKU,UPC Code,Product Description,Brand Code,Brand Name,Status,Sales,Date,YearMonth,PO_Quantity,sales_lag_1,po_quantity_lag_1,sales_lag_2,po_quantity_lag_2,sales_lag_3,po_quantity_lag_3
422,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",APB,APLB,New Seller,79,2024-07-01,2024-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0
661,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",APB,APLB,New Seller,152,2024-08-01,2024-08,0.0,79.0,0.0,0.0,0.0,0.0,0.0
914,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",APB,APLB,New Seller,364,2024-09-01,2024-09,760.0,152.0,0.0,79.0,0.0,0.0,0.0
1165,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",APB,APLB,New Seller,199,2024-10-01,2024-10,440.0,364.0,760.0,152.0,0.0,79.0,0.0
1415,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",APB,APLB,On Sale,308,2024-11-01,2024-11,160.0,199.0,440.0,364.0,760.0,152.0,0.0
1654,APB68267,8809874682679,"Glutathione Niacinamide, Ampoule Serum, 1.35 f...",APB,APLB,On Sale,449,2024-12-01,2024-12,340.0,308.0,160.0,199.0,440.0,364.0,760.0
1892,APB68267,8809874682679,"Glutathione Niacinamide, Ampoule Serum, 1.35 f...",APB,APLB,On Sale,442,2025-01-01,2025-01,560.0,449.0,340.0,308.0,160.0,199.0,440.0
2096,APB68267,8809874682679,"Glutathione Niacinamide, Ampoule Serum, 1.35 f...",APB,APLB,On Sale,340,2025-02-01,2025-02,660.0,442.0,560.0,449.0,340.0,308.0,160.0
2289,APB68267,8809874682679,"Glutathione Niacinamide, Ampoule Serum, 1.35 f...",APB,APLB,On Sale,597,2025-03-01,2025-03,500.0,340.0,660.0,442.0,560.0,449.0,340.0
2522,APB68267,8809874682679,"Glutathione Niacinamide, Ampoule Serum, 1.35 f...",APB,APLB,On Sale,515,2025-04-01,2025-04,560.0,597.0,500.0,340.0,660.0,442.0,560.0



--- df_features의 마지막 10행 ---


,SKU,UPC Code,Product Description,Brand Code,Brand Name,Status,Sales,Date,YearMonth,PO_Quantity,sales_lag_1,po_quantity_lag_1,sales_lag_2,po_quantity_lag_2,sales_lag_3,po_quantity_lag_3
4641,TWA31352,8800276313529,"Glaze Bouncing Tint, 08 Rose Moon, 4.5 g",TWA,2aN,New Seller,6,2025-08-01,2025-08,0.0,0.0,0.0,0.0,180.0,0.0,0.0
3695,TWA31353,8800276313536,"Glaze Bouncing Tint, 09 Candy Chew, 4.5 g",TWA,2aN,New Pending,0,2025-06-01,2025-06,180.0,0.0,0.0,0.0,0.0,0.0,0.0
4160,TWA31353,8800276313536,"Glaze Bouncing Tint, 09 Candy Chew, 4.5 g",TWA,2aN,New Seller,0,2025-07-01,2025-07,0.0,0.0,180.0,0.0,0.0,0.0,0.0
4642,TWA31353,8800276313536,"Glaze Bouncing Tint, 09 Candy Chew, 4.5 g",TWA,2aN,New Seller,8,2025-08-01,2025-08,0.0,0.0,0.0,0.0,180.0,0.0,0.0
3696,TWA31354,8800276313543,"Glaze Bouncing Tint, 10 Cherry Bite, 4.5 g",TWA,2aN,New Pending,0,2025-06-01,2025-06,180.0,0.0,0.0,0.0,0.0,0.0,0.0
4161,TWA31354,8800276313543,"Glaze Bouncing Tint, 10 Cherry Bite, 4.5 g",TWA,2aN,New Seller,1,2025-07-01,2025-07,0.0,0.0,180.0,0.0,0.0,0.0,0.0
4643,TWA31354,8800276313543,"Glaze Bouncing Tint, 10 Cherry Bite, 4.5 g",TWA,2aN,New Seller,9,2025-08-01,2025-08,0.0,1.0,0.0,0.0,180.0,0.0,0.0
3697,TWA60548,8809738605486,"Easy Off Mascara Remover, 0.24 oz (7 g)",TWA,2aN,New Pending,0,2025-06-01,2025-06,180.0,0.0,0.0,0.0,0.0,0.0,0.0
4162,TWA60548,8809738605486,"Easy Off Mascara Remover, 0.24 oz (7 g)",TWA,2aN,New Pending,0,2025-07-01,2025-07,180.0,0.0,180.0,0.0,0.0,0.0,0.0
4644,TWA60548,8809738605486,"Easy Off Mascara Remover, 0.24 oz (7 g)",TWA,2aN,New Pending,0,2025-08-01,2025-08,0.0,0.0,180.0,0.0,180.0,0.0,0.0


In [ ]:
# 3개월 이동 평균 피처 생성
for window in [3]:
    # 과거 판매량의 이동 평균
    df_features[f'sales_rolling_mean_{window}'] = grouped['Sales'].shift(1).rolling(window).mean()
    # 과거 PO 수량의 이동 평균
    df_features[f'po_quantity_rolling_mean_{window}'] = grouped['PO_Quantity'].shift(1).rolling(window).mean()

# 생성된 빈 값(NaN)을 0으로 채움
df_features.fillna(0, inplace=True)

print("✅ 이동 평균 피처 생성이 완료되었습니다.")
print(df_features[['Date', 'SKU', 'Sales', 'sales_rolling_mean_3']].tail())

✅ 이동 평균 피처 생성이 완료되었습니다.
           Date       SKU  Sales  sales_rolling_mean_3
4161 2025-07-01  TWA31354      1                   0.0
4643 2025-08-01  TWA31354      9                   0.0
3697 2025-06-01  TWA60548      0                   0.0
4162 2025-07-01  TWA60548      0                   0.0
4644 2025-08-01  TWA60548      0                   0.0


In [ ]:
df_features

,SKU,UPC Code,Product Description,Brand Code,Brand Name,Status,Sales,Date,YearMonth,PO_Quantity,sales_lag_1,po_quantity_lag_1,sales_lag_2,po_quantity_lag_2,sales_lag_3,po_quantity_lag_3,sales_rolling_mean_3,po_quantity_rolling_mean_3
422,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",APB,APLB,New Seller,79,2024-07-01,2024-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
661,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",APB,APLB,New Seller,152,2024-08-01,2024-08,0.0,79.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
914,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",APB,APLB,New Seller,364,2024-09-01,2024-09,760.0,152.0,0.0,79.0,0.0,0.0,0.0,0.000000,0.000000
1165,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",APB,APLB,New Seller,199,2024-10-01,2024-10,440.0,364.0,760.0,152.0,0.0,79.0,0.0,198.333333,253.333333
1415,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",APB,APLB,On Sale,308,2024-11-01,2024-11,160.0,199.0,440.0,364.0,760.0,152.0,0.0,238.333333,400.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4161,TWA31354,8800276313543,"Glaze Bouncing Tint, 10 Cherry Bite, 4.5 g",TWA,2aN,New Seller,1,2025-07-01,2025-07,0.0,0.0,180.0,0.0,0.0,0.0,0.0,0.000000,0.000000
4643,TWA31354,8800276313543,"Glaze Bouncing Tint, 10 Cherry Bite, 4.5 g",TWA,2aN,New Seller,9,2025-08-01,2025-08,0.0,1.0,0.0,0.0,180.0,0.0,0.0,0.000000,0.000000
3697,TWA60548,8809738605486,"Easy Off Mascara Remover, 0.24 oz (7 g)",TWA,2aN,New Pending,0,2025-06-01,2025-06,180.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
4162,TWA60548,8809738605486,"Easy Off Mascara Remover, 0.24 oz (7 g)",TWA,2aN,New Pending,0,2025-07-01,2025-07,180.0,0.0,180.0,0.0,0.0,0.0,0.0,0.000000,0.000000


In [ ]:
# 원-핫 인코딩을 적용할 컬럼 선택
categorical_cols = ['Status', 'Brand Code']

# 원-핫 인코딩 실행
df_final = pd.get_dummies(df_features, columns=categorical_cols, drop_first=True)

print("\n✅ 범주형 피처 변환이 완료되었습니다.")
print("새로 생성된 컬럼들을 확인하세요:")

df_final = df_final.drop(columns=['Brand Name'])
print("✅ 'Brand Name' 컬럼이 삭제되었습니다.")

# 'YearMonth' 컬럼을 삭제합니다.
df_final = df_final.drop(columns=['YearMonth'])

print("✅ 'YearMonth' 컬럼이 삭제되었습니다.")


print("\n--- 삭제 후 컬럼 목록 ---")
print(df_final.columns)


✅ 범주형 피처 변환이 완료되었습니다.
새로 생성된 컬럼들을 확인하세요:
✅ 'Brand Name' 컬럼이 삭제되었습니다.
✅ 'YearMonth' 컬럼이 삭제되었습니다.

--- 삭제 후 컬럼 목록 ---
Index(['SKU', 'UPC Code', 'Product Description', 'Sales', 'Date',
       'PO_Quantity', 'sales_lag_1', 'po_quantity_lag_1', 'sales_lag_2',
       'po_quantity_lag_2', 'sales_lag_3', 'po_quantity_lag_3',
       'sales_rolling_mean_3', 'po_quantity_rolling_mean_3',
       'Status_New Pending', 'Status_New Seller', 'Status_NoPO',
       'Status_On Sale', 'Brand Code_APL', 'Brand Code_BBU', 'Brand Code_DMX',
       'Brand Code_EQQ', 'Brand Code_GBL', 'Brand Code_GWU', 'Brand Code_HBF',
       'Brand Code_HKI', 'Brand Code_LFT', 'Brand Code_MUZ', 'Brand Code_MVY',
       'Brand Code_NRP', 'Brand Code_OOT', 'Brand Code_PIT', 'Brand Code_PNA',
       'Brand Code_SGE', 'Brand Code_SRI', 'Brand Code_TSN', 'Brand Code_TWA'],
      dtype='object')


In [ ]:
print("--- df_final의 처음 10행 ---")
display(df_final.head(10))

print("\n--- df_final의 마지막 10행 ---")
display(df_final.tail(10))

--- df_final의 처음 10행 ---


,SKU,UPC Code,Product Description,Sales,Date,PO_Quantity,sales_lag_1,po_quantity_lag_1,sales_lag_2,po_quantity_lag_2,sales_lag_3,po_quantity_lag_3,sales_rolling_mean_3,po_quantity_rolling_mean_3,Status_New Pending,Status_New Seller,Status_NoPO,Status_On Sale,Brand Code_APL,Brand Code_BBU,Brand Code_DMX,Brand Code_EQQ,Brand Code_GBL,Brand Code_GWU,Brand Code_HBF,Brand Code_HKI,Brand Code_LFT,Brand Code_MUZ,Brand Code_MVY,Brand Code_NRP,Brand Code_OOT,Brand Code_PIT,Brand Code_PNA,Brand Code_SGE,Brand Code_SRI,Brand Code_TSN,Brand Code_TWA
422,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",79,2024-07-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
661,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",152,2024-08-01,0.0,79.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
914,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",364,2024-09-01,760.0,152.0,0.0,79.0,0.0,0.0,0.0,0.000000,0.000000,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1165,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",199,2024-10-01,440.0,364.0,760.0,152.0,0.0,79.0,0.0,198.333333,253.333333,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1415,APB68267,8809874682679,"Glutathione Niacinamide Ampoule Serum , 1.35 f...",308,2024-11-01,160.0,199.0,440.0,364.0,760.0,152.0,0.0,238.333333,400.000000,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1654,APB68267,8809874682679,"Glutathione Niacinamide, Ampoule Serum, 1.35 f...",449,2024-12-01,340.0,308.0,160.0,199.0,440.0,364.0,760.0,290.333333,453.333333,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1892,APB68267,8809874682679,"Glutathione Niacinamide, Ampoule Serum, 1.35 f...",442,2025-01-01,560.0,449.0,340.0,308.0,160.0,199.0,440.0,318.666667,313.333333,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2096,APB68267,8809874682679,"Glutathione Niacinamide, Ampoule Serum, 1.35 f...",340,2025-02-01,660.0,442.0,560.0,449.0,340.0,308.0,160.0,399.666667,353.333333,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2289,APB68267,8809874682679,"Glutathione Niacinamide, Ampoule Serum, 1.35 f...",597,2025-03-01,500.0,340.0,660.0,442.0,560.0,449.0,340.0,410.333333,520.000000,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2522,APB68267,8809874682679,"Glutathione Niacinamide, Ampoule Serum, 1.35 f...",515,2025-04-01,560.0,597.0,500.0,340.0,660.0,442.0,560.0,459.666667,573.333333,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False



--- df_final의 마지막 10행 ---


,SKU,UPC Code,Product Description,Sales,Date,PO_Quantity,sales_lag_1,po_quantity_lag_1,sales_lag_2,po_quantity_lag_2,sales_lag_3,po_quantity_lag_3,sales_rolling_mean_3,po_quantity_rolling_mean_3,Status_New Pending,Status_New Seller,Status_NoPO,Status_On Sale,Brand Code_APL,Brand Code_BBU,Brand Code_DMX,Brand Code_EQQ,Brand Code_GBL,Brand Code_GWU,Brand Code_HBF,Brand Code_HKI,Brand Code_LFT,Brand Code_MUZ,Brand Code_MVY,Brand Code_NRP,Brand Code_OOT,Brand Code_PIT,Brand Code_PNA,Brand Code_SGE,Brand Code_SRI,Brand Code_TSN,Brand Code_TWA
4641,TWA31352,8800276313529,"Glaze Bouncing Tint, 08 Rose Moon, 4.5 g",6,2025-08-01,0.0,0.0,0.0,0.0,180.0,0.0,0.0,0.0,0.0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
3695,TWA31353,8800276313536,"Glaze Bouncing Tint, 09 Candy Chew, 4.5 g",0,2025-06-01,180.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
4160,TWA31353,8800276313536,"Glaze Bouncing Tint, 09 Candy Chew, 4.5 g",0,2025-07-01,0.0,0.0,180.0,0.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
4642,TWA31353,8800276313536,"Glaze Bouncing Tint, 09 Candy Chew, 4.5 g",8,2025-08-01,0.0,0.0,0.0,0.0,180.0,0.0,0.0,0.0,0.0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
3696,TWA31354,8800276313543,"Glaze Bouncing Tint, 10 Cherry Bite, 4.5 g",0,2025-06-01,180.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
4161,TWA31354,8800276313543,"Glaze Bouncing Tint, 10 Cherry Bite, 4.5 g",1,2025-07-01,0.0,0.0,180.0,0.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
4643,TWA31354,8800276313543,"Glaze Bouncing Tint, 10 Cherry Bite, 4.5 g",9,2025-08-01,0.0,1.0,0.0,0.0,180.0,0.0,0.0,0.0,0.0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
3697,TWA60548,8809738605486,"Easy Off Mascara Remover, 0.24 oz (7 g)",0,2025-06-01,180.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
4162,TWA60548,8809738605486,"Easy Off Mascara Remover, 0.24 oz (7 g)",0,2025-07-01,180.0,0.0,180.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
4644,TWA60548,8809738605486,"Easy Off Mascara Remover, 0.24 oz (7 g)",0,2025-08-01,0.0,0.0,180.0,0.0,180.0,0.0,0.0,0.0,0.0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True


## **Model 학습을 위한 Data to CSV**

In [ ]:

# 저장할 파일 경로와 이름 정의
# 찾기 쉽도록 Colab Notebooks 폴더 안에 저장하는 것을 추천합니다.
output_path = '/content/drive/MyDrive/Colab Notebooks/Data for Modeling/df_for_modeling.csv'

# 데이터프레임을 CSV 파일로 저장
# index=False 옵션은 불필요한 인덱스 컬럼이 파일에 저장되는 것을 방지합니다. (중요)
df_final.to_csv(output_path, index=False)

print(f"✅ 데이터가 성공적으로 저장되었습니다.")
print(f"   경로: {output_path}")

✅ 데이터가 성공적으로 저장되었습니다.
   경로: /content/drive/MyDrive/Colab Notebooks/Data for Modeling/df_for_modeling.csv
